In [1]:
!pip install -q groq
print('✅ Packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.3/806.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.2.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.2.0 which is incompatible.
✅ Packages installed


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

Mounted at /content/drive
✅ Google Drive mounted


In [3]:
class AgenticAI:
    """
    Multi-turn Agentic AI agent using Grok (via Groq API).

    How it works:
    - Receives a pre-configured Groq client (created externally with the API key)
    - Keeps a full conversation history so the model remembers context across turns
    - Supports a customisable system prompt to set the agent's personality/role

    The API key is NOT handled here. It is applied on the website.
    """

    def __init__(self, client, system_prompt='You are a helpful AI assistant.'):
        """
        Args:
            client        : A pre-configured groq.Groq instance
            system_prompt : Instruction that defines the agent's behaviour
        """
        self.client = client
        self.system_prompt = system_prompt
        self.history = []

    def chat(self, user_message):
        """
        Send a message to the agent and receive a reply.
        Conversation history is maintained automatically.

        Args:
            user_message : The user's input text

        Returns:
            str : The agent's reply
        """
        self.history.append({'role': 'user', 'content': user_message})

        # Build messages list: system prompt + full conversation history
        messages = [{'role': 'system', 'content': self.system_prompt}] + self.history

        response = self.client.chat.completions.create(
            model='llama-3.3-70b-versatile',  # Groq-hosted model; alternatives: 'mixtral-8x7b-32768', 'gemma2-9b-it'
            messages=messages
        )
        reply = response.choices[0].message.content

        self.history.append({'role': 'assistant', 'content': reply})
        return reply

    def reset(self):
        """Clear conversation history and start a fresh session."""
        self.history = []


print('✅ AgenticAI class defined')


✅ AgenticAI class defined


In [8]:
import os

SAVE_DIR = '/content/drive/MyDrive/AI_Project/Agentic_Agent'
os.makedirs(SAVE_DIR, exist_ok=True)

agent_code = '''
from groq import Groq


class AgenticAI:
    """
    Multi-turn Agentic AI agent using Grok via Groq API.
    API key is NOT stored here — it is applied externally on the website.
    """

    def __init__(self, client, system_prompt="You are a helpful AI assistant."):
        self.client = client
        self.system_prompt = system_prompt
        self.history = []

    def chat(self, user_message):
        self.history.append({"role": "user", "content": user_message})

        messages = [{"role": "system", "content": self.system_prompt}] + self.history

        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )
        reply = response.choices[0].message.content

        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.history = []
'''

save_path = os.path.join(SAVE_DIR, 'agentic_ai.py')
with open(save_path, 'w') as f:
    f.write(agent_code)

print(f'✅ Agent code saved to: {save_path}')


✅ Agent code saved to: /content/drive/MyDrive/AI_Project/Agentic_Agent/agentic_ai.py


---

## 🌐 How the API Key is Applied — Website Only

The Groq API key never touches this notebook.  
Here is the exact flow on the website:

```
Website Flow (index.html)
─────────────────────────────────────────────────────
 1. User opens the website in their browser
 2. User types their Groq API key into the key input box
 3. User clicks "Save Key"  →  key saved in a JS variable (browser memory)
 4. User types a message and clicks Send
 5. Browser calls Groq API directly:
      POST api.groq.com/openai/v1/chat/completions
           Authorization: Bearer <key from browser memory>
 6. Grok replies  →  response shown in chat
 7. User closes tab  →  key is gone, nothing persisted anywhere
─────────────────────────────────────────────────────
```

This notebook only defines the **agent class and logic**.  
The API key is 100% the website's responsibility.


## Next Steps: Using the AgenticAI Class

Now that the `AgenticAI` class has been defined and saved, you can import it and use it to create an agent. Remember, the `client` object you pass to the `AgenticAI` constructor should be a pre-configured `google.generativeai` client, typically initialized with your API key.

Here's how you can import the class and set up a basic agent for testing purposes.

In [5]:
# Add the SAVE_DIR to the Python path so we can import modules from it
import sys
sys.path.append(SAVE_DIR)

# Now, import the AgenticAI class
from agentic_ai import AgenticAI

print('✅ AgenticAI class imported successfully.')

✅ AgenticAI class imported successfully.


### Initialize the Groq Client

Before creating an `AgenticAI` instance, you need to set up the Groq client.  
Store your `GROQ_API_KEY` in Colab's **Secrets** panel (🔑 icon on the left sidebar).

Get your free API key at: https://console.groq.com


In [6]:
# Import the Groq SDK and configure the API key
from groq import Groq
from google.colab import userdata

# Ensure GROQ_API_KEY is stored in Colab's secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Initialize the Groq client
grok_client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client configured with 'llama-3.3-70b-versatile'.")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

### Create and Interact with the AgenticAI Agent

Now, let's create an instance of our `AgenticAI` and have a short conversation.

In [7]:
# Create an instance of AgenticAI using the Groq client
my_agent = AgenticAI(client=grok_client, system_prompt='You are a friendly and enthusiastic assistant.')

# Start a conversation
print("Agent: Hi there! How can I help you today?")

user_input = "What's the weather like in London?"
agent_response = my_agent.chat(user_input)
print(f"User: {user_input}")
print(f"Agent: {agent_response}")

user_input = "And in Paris?"
agent_response = my_agent.chat(user_input)
print(f"User: {user_input}")
print(f"Agent: {agent_response}")

# Reset the conversation history
my_agent.reset()
print("\nAgent conversation reset.")

user_input = "Tell me a fun fact."
agent_response = my_agent.chat(user_input)
print(f"User: {user_input}")
print(f"Agent: {agent_response}")


NameError: name 'grok_client' is not defined